# Fine-tune PhoBERT for Vietnamese Sentiment Analysis (3-Class)

**Objective**: Fine-tune `vinai/phobert-base-v2` for 3-class sentiment classification with high accuracy.

**Optimizations**:
- Focal Loss for hard example mining
- Class weights for imbalanced data
- Data augmentation with supplementary data
- Linear warmup + decay scheduler
- FP16 mixed precision training

**Target**: >= 90% accuracy on test examples

## 1. Setup Environment

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate scikit-learn seaborn matplotlib
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Check GPU availability
import torch

print("="*50)
print("GPU CHECK")
print("="*50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Available: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.2f} GB")
else:
    print("WARNING: No GPU detected! Training will be slow.")
    print("Go to Runtime -> Change runtime type -> GPU")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# Mount Google Drive for saving model
from google.colab import drive
drive.mount('/content/drive')

# Create output directory
import os
OUTPUT_DIR = '/content/drive/MyDrive/DO_AN_1_main/phobert-sentiment-3class'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F

# Set random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("All libraries imported successfully!")

## 2. Load Dataset and Remap Labels

In [ ]:
# Load dataset from Hugging Face
print("Loading dataset from Hugging Face...")
dataset = load_dataset("anotherpolarbear/vietnamese-sentiment-analysis")

print(f"\nDataset structure:")
print(dataset)
print(f"\nColumns: {dataset['train'].column_names}")
print(f"\nSample data:")
for i in range(3):
    print(f"  [{i}] Label: {dataset['train'][i]['label']}, Text: {dataset['train'][i]['comment'][:80]}...")

In [ ]:
# Define label mapping function
# Original: 1-5 stars
# Target: 0 (negative), 1 (positive), 2 (neutral)

def remap_label(example):
    """Remap 5-star rating to 3-class sentiment.
    
    1-2 stars -> 0 (negative)
    3 stars   -> 2 (neutral)
    4-5 stars -> 1 (positive)
    """
    original_label = example['label']
    
    if original_label in [1, 2]:
        new_label = 0  # negative
    elif original_label == 3:
        new_label = 2  # neutral
    elif original_label in [4, 5]:
        new_label = 1  # positive
    else:
        # Handle edge cases (0-indexed labels)
        if original_label == 0:
            new_label = 0  # negative
        else:
            new_label = 1  # positive
    
    return {'label': new_label, 'text': example['comment']}

# Apply remapping
print("Remapping labels...")
dataset = dataset.map(remap_label, remove_columns=['comment'])

# Check label distribution
train_labels = dataset['train']['label']
label_counts = Counter(train_labels)

LABEL_NAMES = {0: 'NEGATIVE', 1: 'POSITIVE', 2: 'NEUTRAL'}

print("\nLabel distribution after remapping:")
for label, count in sorted(label_counts.items()):
    print(f"  {LABEL_NAMES[label]} ({label}): {count} samples ({count/len(train_labels)*100:.1f}%)")

## 3. Add Supplementary Data

In [ ]:
# Supplementary data for better coverage

# 30 NEGATIVE examples (strong negative, scam, fake products, poor quality)
NEGATIVE_SAMPLES = [
    # Scam/Fraud
    "Shop lua dao, dat hang roi khong giao, mat tien oan",
    "Cua hang lua dao khach hang, san pham gia mao",
    "Bi lua mat tien, shop khong tra loi tin nhan",
    "Hang gia, hang nhai, lua dao nguoi mua",
    "Shop scam, da bao cao len san",
    "Lua dao trang tron, khong co hang ma van thu tien",
    "Mat tien oan vi tin shop nay, hoan toan la lua dao",
    "Shop ban hang gia, lua dao khach hang",
    
    # Poor quality
    "Hang kem chat luong, khong dung mo ta",
    "Chat luong te, khong dang dong tien",
    "San pham loi, hu hong ngay khi nhan",
    "Hang nhan ve bi vo, chat luong qua te",
    "Khong giong hinh, chat luong kem",
    "Hang rac, khong the su dung duoc",
    "Chat lieu mong manh, de hu",
    "San pham gia re nhung chat luong con te hon",
    
    # Bad service
    "Shop thai do te, khong ho tro khach hang",
    "Nhan vien tu van vo trach nhiem",
    "Dich vu cham, khong tra loi tin nhan",
    "Shop khong giai quyet khieu nai",
    "Bao hanh khong dung cam ket",
    "Khong the lien lac voi shop",
    
    # Shipping issues
    "Giao hang qua cham, doi hon 1 thang",
    "Hang bi hu hong do van chuyen, shop khong den bu",
    "Dong goi cau tha, hang bi bop meo",
    "Ship sai dia chi, mat hang",
    
    # Strong dissatisfaction
    "Tham hoa, khong bao gio mua lai",
    "Qua that vong, phi tien",
    "Te nhat tung thay, 0 sao",
    "Khong dang 1 dong, hoan tien ngay"
]

# 20 POSITIVE examples (short, casual positive reviews)
POSITIVE_SAMPLES = [
    # Quality
    "Chat luong tot",
    "Hang dep lam",
    "San pham tuyet voi",
    "Chat luong vuot mong doi",
    "Hang xin, dung mo ta",
    
    # Price
    "Gia re ma chat",
    "Dang dong tien",
    "Gia hop ly, chat luong tot",
    "Re ma tot",
    
    # Repurchase intent
    "Se mua lai",
    "Chac chan quay lai ung ho",
    "Mua lan 2 roi, van tot",
    "Ung ho shop dai dai",
    
    # Satisfaction
    "Rat hai long",
    "Ua thich lam",
    "Oki lun",
    "10 diem",
    "5 sao",
    "Tuyet voi",
    "Xuat sac"
]

# 15 NEUTRAL examples (clear neutral statements)
NEUTRAL_SAMPLES = [
    "Hang binh thuong, khong tot khong xau",
    "Tam duoc, chua co gi dac biet",
    "Dung nhu mo ta, khong hon khong kem",
    "OK",
    "Binh thuong",
    "Chua dung nen chua danh gia duoc",
    "Se danh gia sau khi su dung",
    "Giao hang dung hen",
    "Dong goi can than",
    "Hang nhan duoc, chua mo",
    "Moi nhan, chua test",
    "Tui den roi",
    "Da nhan hang",
    "Hang ve nhanh",
    "Ship nhanh"
]

print(f"Supplementary data:")
print(f"  NEGATIVE: {len(NEGATIVE_SAMPLES)} samples")
print(f"  POSITIVE: {len(POSITIVE_SAMPLES)} samples")
print(f"  NEUTRAL: {len(NEUTRAL_SAMPLES)} samples")

In [ ]:
# Create supplementary dataset with augmentation (repeat 4 times)
AUGMENTATION_FACTOR = 4

supplementary_data = []

for _ in range(AUGMENTATION_FACTOR):
    for text in NEGATIVE_SAMPLES:
        supplementary_data.append({'text': text, 'label': 0})
    for text in POSITIVE_SAMPLES:
        supplementary_data.append({'text': text, 'label': 1})
    for text in NEUTRAL_SAMPLES:
        supplementary_data.append({'text': text, 'label': 2})

supplementary_df = pd.DataFrame(supplementary_data)
supplementary_dataset = Dataset.from_pandas(supplementary_df)

print(f"Supplementary dataset after {AUGMENTATION_FACTOR}x augmentation: {len(supplementary_dataset)} samples")
print(f"  NEGATIVE: {len(NEGATIVE_SAMPLES) * AUGMENTATION_FACTOR}")
print(f"  POSITIVE: {len(POSITIVE_SAMPLES) * AUGMENTATION_FACTOR}")
print(f"  NEUTRAL: {len(NEUTRAL_SAMPLES) * AUGMENTATION_FACTOR}")

In [ ]:
# Combine original and supplementary data
print(f"Original train size: {len(dataset['train'])}")

# Concatenate datasets
combined_train = concatenate_datasets([dataset['train'], supplementary_dataset])

print(f"Combined train size: {len(combined_train)}")

# Shuffle the combined dataset
combined_train = combined_train.shuffle(seed=SEED)

# Check new label distribution
new_label_counts = Counter(combined_train['label'])
print("\nNew label distribution:")
for label, count in sorted(new_label_counts.items()):
    print(f"  {LABEL_NAMES[label]} ({label}): {count} samples ({count/len(combined_train)*100:.1f}%)")

## 4. Compute Class Weights

In [ ]:
# Compute class weights using sklearn
train_labels = np.array(combined_train['label'])
classes = np.array([0, 1, 2])

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=train_labels
)

class_weights_dict = {i: w for i, w in enumerate(class_weights)}
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("Class weights (balanced):")
for label, weight in class_weights_dict.items():
    print(f"  {LABEL_NAMES[label]} ({label}): {weight:.4f}")

# Move to GPU if available
if torch.cuda.is_available():
    class_weights_tensor = class_weights_tensor.cuda()
    print(f"\nClass weights moved to GPU")

## 5. Tokenization

In [ ]:
# Load PhoBERT tokenizer
MODEL_NAME = "vinai/phobert-base-v2"
MAX_LENGTH = 128

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenizer loaded successfully!")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Max length: {MAX_LENGTH}")

In [ ]:
# Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors=None
    )

# Tokenize all datasets
print("Tokenizing datasets...")

tokenized_train = combined_train.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing train"
)

# Check if test split exists, otherwise use validation
if 'test' in dataset:
    tokenized_test = dataset['test'].map(
        tokenize_function,
        batched=True,
        desc="Tokenizing test"
    )
elif 'validation' in dataset:
    tokenized_test = dataset['validation'].map(
        tokenize_function,
        batched=True,
        desc="Tokenizing validation"
    )
else:
    # Split train into train/test
    split = tokenized_train.train_test_split(test_size=0.1, seed=SEED)
    tokenized_train = split['train']
    tokenized_test = split['test']

print(f"\nTokenized train size: {len(tokenized_train)}")
print(f"Tokenized test size: {len(tokenized_test)}")

# Set format for PyTorch
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

## 6. Load Model

In [ ]:
# Load PhoBERT model for sequence classification
NUM_LABELS = 3

print(f"Loading model: {MODEL_NAME}")
print(f"Number of labels: {NUM_LABELS}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="single_label_classification"
)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"\nModel loaded successfully!")
print(f"Device: {device}")
print(f"Model parameters: {model.num_parameters():,}")

## 7. Custom Trainer with Focal Loss

In [ ]:
# Define Focal Loss
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance and hard examples.
    
    Focal Loss = -alpha * (1 - pt)^gamma * log(pt)
    
    where pt is the probability of the correct class.
    
    Args:
        alpha: Class weights tensor
        gamma: Focusing parameter (default=2.0)
        reduction: 'mean', 'sum', or 'none'
    """
    
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        # Compute softmax probabilities
        probs = F.softmax(inputs, dim=1)
        
        # Get probability of correct class
        targets_one_hot = F.one_hot(targets, num_classes=inputs.size(1)).float()
        pt = (probs * targets_one_hot).sum(dim=1)
        
        # Compute focal weight
        focal_weight = (1 - pt) ** self.gamma
        
        # Compute cross entropy
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        
        # Apply focal weight
        focal_loss = focal_weight * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

print("Focal Loss defined successfully!")

In [ ]:
# Custom Trainer with Focal Loss
class FocalLossTrainer(Trainer):
    """Custom Trainer that uses Focal Loss instead of CrossEntropyLoss."""
    
    def __init__(self, *args, class_weights=None, focal_gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_gamma = focal_gamma
        self.focal_loss_fn = None
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Initialize Focal Loss with class weights on correct device
        if self.focal_loss_fn is None:
            if self.class_weights is not None:
                weights = self.class_weights.to(logits.device)
            else:
                weights = None
            self.focal_loss_fn = FocalLoss(
                alpha=weights,
                gamma=self.focal_gamma,
                reduction='mean'
            )
        
        loss = self.focal_loss_fn(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

print("FocalLossTrainer defined successfully!")

In [ ]:
# Define evaluation metrics
def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, and F1 score."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

print("Evaluation metrics defined successfully!")

## 8. Training

In [ ]:
# Training arguments - optimized for GPU T4
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01

training_args = TrainingArguments(
    output_dir='./results',
    
    # Training hyperparameters
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    
    # Optimizer settings
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type='linear',
    
    # Evaluation and logging
    eval_strategy='steps',
    eval_steps=200,
    logging_dir='./logs',
    logging_steps=100,
    
    # Saving
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    
    # Performance optimization
    fp16=True,
    dataloader_num_workers=2,
    
    # Misc
    seed=SEED,
    report_to='none',
    disable_tqdm=False
)

print("Training Arguments:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Warmup steps: {WARMUP_STEPS}")
print(f"  FP16: True")

In [ ]:
# Initialize trainer with Focal Loss
trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
    focal_gamma=2.0,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Trainer initialized with Focal Loss!")
print(f"Focal gamma: 2.0")
print(f"Early stopping patience: 3")

In [ ]:
# Start training
print("="*60)
print("STARTING TRAINING")
print("="*60)

train_result = trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Training samples/second: {train_result.metrics['train_samples_per_second']:.2f}")

## 9. Evaluation

In [ ]:
# Evaluate on test set
print("="*60)
print("EVALUATION ON TEST SET")
print("="*60)

eval_results = trainer.evaluate()

print(f"\nEvaluation Results:")
print(f"  Accuracy:  {eval_results['eval_accuracy']*100:.2f}%")
print(f"  Precision: {eval_results['eval_precision']*100:.2f}%")
print(f"  Recall:    {eval_results['eval_recall']*100:.2f}%")
print(f"  F1 (macro): {eval_results['eval_f1']*100:.2f}%")

In [ ]:
# Get predictions for confusion matrix
predictions_output = trainer.predict(tokenized_test)
predictions = np.argmax(predictions_output.predictions, axis=-1)
true_labels = predictions_output.label_ids

# Confusion Matrix
cm = confusion_matrix(true_labels, predictions)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['NEGATIVE', 'POSITIVE', 'NEUTRAL'],
    yticklabels=['NEGATIVE', 'POSITIVE', 'NEUTRAL']
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - PhoBERT 3-Class Sentiment')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png', dpi=150)
plt.show()

print(f"Confusion matrix saved to {OUTPUT_DIR}/confusion_matrix.png")

In [ ]:
# Detailed Classification Report
print("="*60)
print("CLASSIFICATION REPORT")
print("="*60)

report = classification_report(
    true_labels,
    predictions,
    target_names=['NEGATIVE', 'POSITIVE', 'NEUTRAL'],
    digits=4
)
print(report)

# Save report to file
with open(f'{OUTPUT_DIR}/classification_report.txt', 'w') as f:
    f.write("PhoBERT 3-Class Sentiment Classification Report\n")
    f.write("="*60 + "\n\n")
    f.write(report)

print(f"Classification report saved to {OUTPUT_DIR}/classification_report.txt")

## 10. Test with 20 Example Sentences

In [ ]:
# 20 test sentences from easy to hard with expected labels
TEST_SENTENCES = [
    # EASY - Clear sentiment
    ("San pham rat tot, chat luong tuyet voi", 1, "POSITIVE"),  # 1
    ("Hang te qua, khong dung mo ta", 0, "NEGATIVE"),  # 2
    ("Da nhan hang, chua mo", 2, "NEUTRAL"),  # 3
    ("Shop lua dao, mat tien oan", 0, "NEGATIVE"),  # 4
    ("5 sao, se ung ho tiep", 1, "POSITIVE"),  # 5
    
    # MEDIUM - Casual/Short
    ("OK", 2, "NEUTRAL"),  # 6
    ("Tam duoc", 2, "NEUTRAL"),  # 7
    ("Gia re ma tot", 1, "POSITIVE"),  # 8
    ("Chat luong kem", 0, "NEGATIVE"),  # 9
    ("Giao hang nhanh", 2, "NEUTRAL"),  # 10
    
    # MEDIUM-HARD - Mixed signals
    ("Hang dep nhung giao cham", 1, "POSITIVE"),  # 11 - positive despite delay
    ("Gia dat nhung chat luong tot", 1, "POSITIVE"),  # 12 - expensive but good
    ("Ship nhanh, dong goi can than", 2, "NEUTRAL"),  # 13 - neutral (service)
    ("Mau khong giong hinh lam", 0, "NEGATIVE"),  # 14 - different from picture
    ("Chua dung nen chua biet", 2, "NEUTRAL"),  # 15 - haven't used yet
    
    # HARD - Sarcasm/Implicit
    ("Tuyet voi lun, lan sau khong mua nua", 0, "NEGATIVE"),  # 16 - sarcasm
    ("10 diem cho su that vong", 0, "NEGATIVE"),  # 17 - ironic
    ("Mua nhieu lan roi, van tin tuong shop", 1, "POSITIVE"),  # 18 - repeat customer
    ("Khong co gi de che", 1, "POSITIVE"),  # 19 - nothing to complain
    ("Binh thuong, khong dac biet", 2, "NEUTRAL")  # 20 - neutral statement
]

print(f"Total test sentences: {len(TEST_SENTENCES)}")

In [ ]:
# Prediction function
def predict_sentiment(text, model, tokenizer, device):
    """Predict sentiment for a single text."""
    model.eval()
    
    # Tokenize
    inputs = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors='pt'
    )
    
    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][predicted_class].item()
    
    return predicted_class, confidence

print("Prediction function defined!")

In [ ]:
# Test all 20 sentences
print("="*80)
print("TESTING 20 EXAMPLE SENTENCES")
print("="*80)

correct = 0
results = []

for i, (text, expected_label, expected_name) in enumerate(TEST_SENTENCES, 1):
    predicted_label, confidence = predict_sentiment(text, model, tokenizer, device)
    predicted_name = LABEL_NAMES[predicted_label]
    
    is_correct = predicted_label == expected_label
    if is_correct:
        correct += 1
        status = "[CORRECT]"
    else:
        status = "[WRONG]"
    
    results.append({
        'No': i,
        'Text': text,
        'Expected': expected_name,
        'Predicted': predicted_name,
        'Confidence': confidence,
        'Correct': is_correct
    })
    
    print(f"\n[{i:02d}] {text}")
    print(f"     Expected: {expected_name} | Predicted: {predicted_name} ({confidence*100:.1f}%) {status}")

accuracy = correct / len(TEST_SENTENCES) * 100

print("\n" + "="*80)
print(f"FINAL ACCURACY: {correct}/{len(TEST_SENTENCES)} = {accuracy:.1f}%")
print("="*80)

if accuracy >= 90:
    print("TARGET ACHIEVED: >= 90% accuracy")
else:
    print(f"TARGET NOT ACHIEVED: Need {90 - accuracy:.1f}% more")

In [ ]:
# Save test results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv(f'{OUTPUT_DIR}/test_results_20_sentences.csv', index=False)

print(f"Test results saved to {OUTPUT_DIR}/test_results_20_sentences.csv")
print("\nResults DataFrame:")
display(results_df)

## 11. Save Model to Google Drive

In [ ]:
# Save model and tokenizer
print("="*60)
print("SAVING MODEL TO GOOGLE DRIVE")
print("="*60)

# Save model
model.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# Save tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Tokenizer saved to {OUTPUT_DIR}")

# Save training config
config = {
    'model_name': MODEL_NAME,
    'num_labels': NUM_LABELS,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'num_epochs': NUM_EPOCHS,
    'warmup_steps': WARMUP_STEPS,
    'focal_gamma': 2.0,
    'class_weights': class_weights.tolist(),
    'label_mapping': LABEL_NAMES,
    'test_accuracy': accuracy
}

import json
with open(f'{OUTPUT_DIR}/training_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"Training config saved to {OUTPUT_DIR}/training_config.json")

# List saved files
print("\nSaved files:")
for file in os.listdir(OUTPUT_DIR):
    file_path = os.path.join(OUTPUT_DIR, file)
    size = os.path.getsize(file_path) / 1024 / 1024  # MB
    print(f"  {file}: {size:.2f} MB")

In [ ]:
# Final summary
print("\n" + "="*80)
print("TRAINING COMPLETE - SUMMARY")
print("="*80)
print(f"")
print(f"Model: {MODEL_NAME}")
print(f"Task: 3-Class Vietnamese Sentiment Analysis")
print(f"Labels: NEGATIVE (0), POSITIVE (1), NEUTRAL (2)")
print(f"")
print(f"Training Data: {len(tokenized_train)} samples")
print(f"Test Data: {len(tokenized_test)} samples")
print(f"")
print(f"Evaluation Metrics:")
print(f"  - Accuracy:  {eval_results['eval_accuracy']*100:.2f}%")
print(f"  - Precision: {eval_results['eval_precision']*100:.2f}%")
print(f"  - Recall:    {eval_results['eval_recall']*100:.2f}%")
print(f"  - F1 (macro): {eval_results['eval_f1']*100:.2f}%")
print(f"")
print(f"20 Sentence Test: {correct}/{len(TEST_SENTENCES)} = {accuracy:.1f}%")
print(f"")
print(f"Model saved to: {OUTPUT_DIR}")
print("="*80)

## 12. Load and Test Saved Model (Optional)

In [ ]:
# Load saved model for inference
print("Loading saved model for verification...")

loaded_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
loaded_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
loaded_model = loaded_model.to(device)
loaded_model.eval()

print("Model loaded successfully!")

# Quick test
test_texts = [
    "San pham rat tot",
    "Hang kem chat luong",
    "Binh thuong"
]

print("\nQuick verification test:")
for text in test_texts:
    pred, conf = predict_sentiment(text, loaded_model, loaded_tokenizer, device)
    print(f"  '{text}' -> {LABEL_NAMES[pred]} ({conf*100:.1f}%)")